In [1]:
import os
import yaml

# Search for data.yaml anywhere inside Kaggle input
yaml_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "data.yaml" in files:
        yaml_path = os.path.join(root, "data.yaml")
        break

print("Found data.yaml at:", yaml_path)

# Make sure it was actually found
if yaml_path is None:
    print("ERROR: data.yaml was not found.")
else:
    # Read dataset configuration
    with open(yaml_path, "r") as f:
        data_config = yaml.safe_load(f)

    print("\n--- DATASET DETAILS ---")
    print("Classes:", data_config.get("names"))
    print("Number of classes (nc):", data_config.get("nc"))
    print("Train path:", data_config.get("train"))
    print("Validation path:", data_config.get("val"))

Found data.yaml at: /kaggle/input/datasets/beyzakucuk/ppe-detection-v1/data.yaml

--- DATASET DETAILS ---
Classes: ['boots', 'gloves', 'goggles', 'helmet', 'person', 'vest']
Number of classes (nc): 6
Train path: ../train/images
Validation path: ../valid/images


In [2]:
import os
import yaml

# Base directory where the images actually live
base_dir = os.path.dirname(yaml_path)

# Build a clean dictionary with absolute paths
patched_config = {
    'path': base_dir,
    'train': os.path.join(base_dir, 'train', 'images'),
    'val': os.path.join(base_dir, 'valid', 'images'),
    'test': os.path.join(base_dir, 'test', 'images') if os.path.exists(os.path.join(base_dir, 'test', 'images')) else None,
    'nc': data_config.get('nc', 6),
    'names': data_config.get('names')
}

# Remove test key if test folder does not exist
if patched_config['test'] is None:
    del patched_config['test']

# Save the corrected yaml to your writable working directory
fixed_yaml_path = "/kaggle/working/ppe_fixed_data.yaml"
with open(fixed_yaml_path, 'w') as f:
    yaml.dump(patched_config, f, default_flow_style=False)

print(f"Created fixed configuration at: {fixed_yaml_path}")
with open(fixed_yaml_path, 'r') as f:
    print(f.read())

Created fixed configuration at: /kaggle/working/ppe_fixed_data.yaml
names:
- boots
- gloves
- goggles
- helmet
- person
- vest
nc: 6
path: /kaggle/input/datasets/beyzakucuk/ppe-detection-v1
test: /kaggle/input/datasets/beyzakucuk/ppe-detection-v1/test/images
train: /kaggle/input/datasets/beyzakucuk/ppe-detection-v1/train/images
val: /kaggle/input/datasets/beyzakucuk/ppe-detection-v1/valid/images



In [3]:
import torch

# 1. Check if CUDA (GPU support) is available
print(f"Is GPU available?: {torch.cuda.is_available()}")

# 2. Get the number of available GPUs
device_count = torch.cuda.device_count()
print(f"Number of GPUs found: {device_count}")

# 3. Print the names of the detected GPUs
for i in range(device_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


Is GPU available?: True
Number of GPUs found: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [4]:
# 1. Install Ultralytics
!pip install -q ultralytics

from ultralytics import YOLO

# 2. Load pre-trained nano weights (weights from COCO)
model = YOLO("yolov8n.pt")

# 3. Train on the fixed configuration
results = model.train(
    data=fixed_yaml_path,
    epochs=20,
    imgsz=640,
    batch=16,
    device="0",      # <--- FIXED: Wrapped in quotes so Kaggle successfully shifts from CPU to GPU
    workers=2,
    project="hazard_vision",
    name="ppe_yolov8n",
    save=True,
    verbose=True
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 4.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.165 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ppe_fixed_data.yaml, degrees=0.

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/20      2.58G      1.145      1.874       1.21         49        640: 100% ━━━━━━━━━━━━ 867/867 5.6it/s 2:360.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 5.5it/s 9.5s0.2s
                   all       1656       6258        0.4      0.485      0.362      0.232

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/20      2.58G      1.149      1.724      1.214         66        640: 100% ━━━━━━━━━━━━ 867/867 5.5it/s 2:360.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 5.4it/s 9.5s0.2s
                   all       1656       6258      0.455      0.502       0.39      0.247

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/20      2.58G      1.125      1.615      1.194         59        640: 100% ━━━━━━━━━━━━ 867/867 5.6it/s 2:340.4sss
                 Class     Imag

In [5]:
from ultralytics import YOLO

# Load the best checkpoint from the completed run
best_pt_path = "/kaggle/working/runs/detect/hazard_vision/ppe_yolov8n/weights/best.pt"
model = YOLO(best_pt_path)

# Export to ONNX:
# - dynamic=False: static shape (640x640) optimizes engine execution on CPU
# - simplify=True: cleans redundant nodes
onnx_path = model.export(format="onnx", imgsz=640, dynamic=False, simplify=True)
print(f"\n[DONE] ONNX Model exported to: {onnx_path}")

Ultralytics 8.4.165 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/hazard_vision/ppe_yolov8n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 405ms
Prepared 2 packages in 388ms
Installed 2 packages in 13ms
 + onnxruntime==1.30.0
 + onnxslim==0.1.97

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export